# Optimisation par gradient

In [1]:
import meep as mp
import meep.adjoint as mpa
import numpy as np
from autograd import numpy as npa
from matplotlib import pyplot as plt
import math
import nlopt

## Paramètres

In [2]:
# --- Paramètres physiques ---
resolution = 100          # pixels/um 
lam_cen = 1.55             # longueur d'onde centrale (um)
theta = math.radians(10)   # angle d'incidence de la fibre par rapport a la normale (rad)
n_core = np.sqrt(12.0)     # indice de refraction du silicium 
Si = mp.Medium(epsilon=12.0)  # materiau silicium 
h = 0.22                   # epaisseur totale du slab silicium (um)
gp = 0.890                 # periode du reseau (um)
gdc = 0.5                  # duty cycle du reseau (fraction de silicium par periode)
gN = 20                    # nombre de dents du reseau
t_slab = 0.07              # hauteur de la partie non gravée du réseau

# --- Domaine ---
dpml = 1.0                 # epaisseur des couches PML (um)
margin_before = 8.0        # marge d'air avant le reseau
dair = 2.0                 # marge d'air au-dessus et en dessous du slab, en um
Lwg = 8.0                  # longueur du guide de sortie monomode (um)
Lgrating = gN * gp         # longueur totale du reseau
sx = dpml + margin_before + Lgrating + Lwg + dpml   # largeur totale du domaine de simulation 
sy = dpml + dair + h + dair + dpml                   # hauteur totale du domaine de simulation 
cell_size = mp.Vector3(sx, sy, 0)                    # taille de la cellule Meep
x0 = -0.5 * sx + dpml + margin_before                # position x de debut du reseau (bord gauche de la 1ere dent)
x_wg_start = x0 + Lgrating                           # position x ou commence le guide de sortie (juste apres le reseau)
boundary_layers = [mp.PML(dpml)]                     # conditions aux limites absorbantes sur les 4 bords du domaine
fcen = 1 / lam_cen                                   # frequence centrale correspondant a lam_cen (unites Meep : c=1)

# --- Source ---
d_standoff = 1.0            # distance verticale entre la ligne source et la surface du reseau (um)
y_src = h/2 + d_standoff     # position y de la source 
x_focus = x0 + Lgrating/2    # position x du point focal du faisceau gaussien 
beam_kdir = mp.Vector3(math.sin(theta), -math.cos(theta), 0)  # vecteur direction de propagation du faisceau 
beam_x0 = mp.Vector3(0, -d_standoff, 0)              # position du point focal
df = 0.05 * fcen             # largeur de bande de la source pulsee 

# --- Variables ---
t_etch = h - t_slab          # hauteur de la partie gravée du réseau
mon_pt = mp.Vector3(x_wg_start + 0.7*Lwg, 0, 0)

## Script

In [ ]:
Air = mp.Medium(index=1.0)
# --- Sources ---
sources = [mp.GaussianBeam2DSource(
        src=mp.GaussianSource(fcen, fwidth=df),
        center=mp.Vector3(x_focus, y_src, 0), size=mp.Vector3(20, 0, 0),
        beam_x0=mp.Vector3(0, -d_standoff, 0), beam_kdir=beam_kdir,
        beam_w0=5.2, beam_E0=mp.Vector3(0,0,1))]

# --- Region de conception : remplace exactement la zone des dents ---
design_region_resolution = resolution
Nx = int(design_region_resolution * Lgrating)
Ny = int(design_region_resolution * t_etch)

design_variables = mp.MaterialGrid(
    mp.Vector3(Nx, Ny), Air, Si, grid_type="U_MEAN"
)
design_region = mpa.DesignRegion(
    design_variables,
    volume=mp.Volume(
        center=mp.Vector3(x0 + Lgrating/2, -h/2 + t_slab + t_etch/2, 0),
        size=mp.Vector3(Lgrating, t_etch, 0),
    ),
)

# --- Geometrie : base fixe + guide de sortie fixe + region variable (remplace la boucle des dents) ---
geometry = [
    mp.Block(size=mp.Vector3(Lgrating + Lwg + dpml, t_slab, mp.inf),                        # grand rectangle
              center=mp.Vector3(x0 + (Lgrating+Lwg+dpml)/2, -h/2 + t_slab/2, 0),
              material=Si),
    mp.Block(size=mp.Vector3(Lwg + dpml, t_etch, mp.inf),                                    # petit rectangle
              center=mp.Vector3(x_wg_start + (Lwg+dpml)/2, -h/2 + t_slab + t_etch/2, 0),
              material=Si),
    mp.Block(center=design_region.center, size=design_region.size, material=design_variables),  # partie ou il y avait les dents remplacer par region de conception
]

sim = mp.Simulation(resolution=resolution, cell_size=cell_size,
                     boundary_layers=boundary_layers, geometry=geometry, sources=sources)

# --- Fonction objectif : meme grandeur physique que coupled_power, version differentiable ---
obj_list = [mpa.EigenmodeCoefficient(
    sim, mp.Volume(center=mon_pt, size=mp.Vector3(0, sy - 2*dpml, 0)), 1
)]

def J(mode_coeff):
    return npa.abs(mode_coeff) ** 2

opt = mpa.OptimizationProblem(
    simulation=sim,
    objective_functions=J,
    objective_arguments=obj_list,
    design_regions=[design_region],
    fcen=fcen, df=df, nf=1,
)

# --- Optimisation ---
n_params = Nx * Ny
x_init = np.full(n_params, gdc)   # depart : densite uniforme = ton duty cycle optimal trouve avant

def f(x, grad):
    val, dJ_dx = opt([x])
    if grad.size > 0:
        grad[:] = dJ_dx.flatten()
    print("efficacite actuelle:", 10*np.log10(val[0]/incident_power), "dB")
    return float(val[0])

solver = nlopt.opt(nlopt.LD_MMA, n_params)
solver.set_max_objective(f)
solver.set_lower_bounds(0.0)
solver.set_upper_bounds(1.0)
solver.set_maxeval(20)   # commence petit pour valider que ca tourne

x_opt = solver.optimize(x_init)

Starting forward run...
-----------
Initializing structure...
time for choose_chunkdivision = 0.00119519 s
Working in 2D dimensions.
Computational cell is 35.8 x 6.22 x 0 with resolution 100
     block, center = (4.5,-0.075,0)
          size (26.8,0.07,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12,12,12)
     block, center = (13.4,0.035,0)
          size (9,0.15,1e+20)
          axes (1,0,0), (0,1,0), (0,0,1)
          dielectric constant epsilon diagonal = (12,12,12)
     block, center = (1.77636e-15,0.035,0)
          size (17.8,0.15,0)
          axes (1,0,0), (0,1,0), (0,0,1)
time for set_epsilon = 2.75715 s
-----------
on time step 282 (time=1.41), 0.0142085 s/step
on time step 564 (time=2.82), 0.0142162 s/step
on time step 848 (time=4.24), 0.0140991 s/step
on time step 1133 (time=5.665), 0.0140785 s/step
on time step 1419 (time=7.095), 0.0140224 s/step
on time step 1700 (time=8.5), 0.0142444 s/step
on time step 1962 (time=9.81

run 0 finished at t = 805.49 (161098 timesteps)
MPB solved for frequency_1(0.645161,0,0) = 0.310265 after 15 iters
MPB solved for frequency_1(1.68346,0,0) = 0.613912 after 8 iters
MPB solved for frequency_1(1.79535,0,0) = 0.645106 after 7 iters
MPB solved for frequency_1(1.79554,0,0) = 0.645161 after 4 iters
MPB solved for frequency_1(1.79554,0,0) = 0.645161 after 1 iters
Dominant planewave for band 1: (1.795543,-0.000000,0.000000)
Starting adjoint run...


MPB solved for frequency_1(-0.645161,0,0) = 0.310265 after 15 iters
MPB solved for frequency_1(-1.68346,0,0) = 0.613912 after 8 iters
MPB solved for frequency_1(-1.79535,0,0) = 0.645106 after 7 iters
MPB solved for frequency_1(-1.79554,0,0) = 0.645161 after 4 iters
MPB solved for frequency_1(-1.79554,0,0) = 0.645161 after 1 iters
on time step 144 (time=0.72), 0.0279368 s/step
on time step 299 (time=1.495), 0.0259561 s/step
on time step 452 (time=2.26), 0.0262995 s/step
on time step 605 (time=3.025), 0.0262529 s/step
on time step 757 (time=3.785), 0.02639 s/step
on time step 911 (time=4.555), 0.0260139 s/step
on time step 1066 (time=5.33), 0.0259487 s/step
on time step 1220 (time=6.1), 0.0261524 s/step
on time step 1375 (time=6.875), 0.0259304 s/step
on time step 1522 (time=7.61), 0.0272376 s/step
on time step 1649 (time=8.245), 0.0316366 s/step
on time step 1776 (time=8.88), 0.0315588 s/step
on time step 1903 (time=9.515), 0.0315619 s/step
on time step 2030 (time=10.15), 0.0315513 s/st